# Shard merge: Combine all shards into one dataset

**What this does:** takes all `shard_<NAME>.pt` files produced by `shard_collect.ipynb` and concatenates them into one `(X, y, problem_ids)` dataset for the probe trainer.

**Runs on CPU. No GPU needed, no compute units consumed**

## What this saves

- `X.pt` -> merged hidden states, shape `[total_positions, HIDDEN_DIM]`
- `y.pt` -> merged labels, shape `[total_positions]`
- `problem_ids.pt` -> which problem each position came from, shape `[total_positions]`
- `merge_meta.json` -> metadata recording which shards were merged and consistency checks

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q torch pandas

## Config

In [ ]:
# Folder that holds all shard_*.pt files. Default is the same OUTPUT_DIR.
# If teammates sent you their shards separately, put them all in one folder
# and point SHARDS_DIR at it.
SHARDS_DIR = OUTPUT_DIR

# Output paths for the merged dataset
X_PATH           = os.path.join(OUTPUT_DIR, "X.pt")
Y_PATH           = os.path.join(OUTPUT_DIR, "y.pt")
PID_PATH         = os.path.join(OUTPUT_DIR, "problem_ids.pt")
MERGE_META_PATH  = os.path.join(OUTPUT_DIR, "merge_meta.json")

# Require at least this many shards; error otherwise (safety against merging
# with missing shards by accident).
MIN_EXPECTED_SHARDS = 1

In [ ]:
# === Discover shard files ===
import glob

pattern = os.path.join(SHARDS_DIR, "shard_*.pt")
# Exclude _partial checkpoints — those are incomplete
shard_paths = sorted(p for p in glob.glob(pattern) if "_partial" not in p)

print(f"Looking in: {SHARDS_DIR}")
print(f"Found {len(shard_paths)} shard file(s):")
for p in shard_paths:
    print(f"  {os.path.basename(p)}  ({os.path.getsize(p) / 1e6:.1f} MB)")

assert len(shard_paths) >= MIN_EXPECTED_SHARDS, (
    f"Only found {len(shard_paths)} shards; expected at least {MIN_EXPECTED_SHARDS}. "
    "Did all teammates upload their shards?"
)

In [ ]:
# === Load + inspect each shard ===
import torch
import pandas as pd

shards = []
for p in shard_paths:
    s = torch.load(p, map_location="cpu", weights_only=False)
    shards.append((p, s))

rows = []
for p, s in shards:
    m = s.get("meta", {})
    n_pos = s["X"].shape[0]
    n_correct = int(s["y"].sum())
    n_unique_problems = len(torch.unique(s["problem_ids"])) if "problem_ids" in s else None
    rows.append({
        "file": os.path.basename(p),
        "NAME": m.get("NAME", "?"),
        "range": f"{m.get('START_IDX', '?')}..{m.get('END_IDX', '?')}",
        "positions": n_pos,
        "unique_problems": n_unique_problems,
        "correct": n_correct,
        "incorrect": n_pos - n_correct,
        "MAX_NEW_TOKENS": m.get("MAX_NEW_TOKENS"),
        "PROBE_LAYER": m.get("PROBE_LAYER"),
        "HIDDEN_DIM": m.get("HIDDEN_DIM"),
        "MODEL_ID": m.get("MODEL_ID"),
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
# === Consistency checks: shards must share the same protocol ===
def _unique(col):
    vals = [r[col] for r in rows if r[col] is not None]
    return set(vals)

for field in ["MAX_NEW_TOKENS", "PROBE_LAYER", "HIDDEN_DIM", "MODEL_ID"]:
    uniq = _unique(field)
    assert len(uniq) <= 1, (
        f"Inconsistent {field} across shards: {uniq}. "
        "Regenerate any mismatched shards with the shared protocol."
    )
    print(f"  {field}: {uniq.pop() if uniq else '(none recorded)'}")

# Check that no two shards cover the same problem_ids (overlap = double-counted data)
all_pids_per_shard = [set(s["problem_ids"].tolist()) for _, s in shards]
for i in range(len(shards)):
    for j in range(i + 1, len(shards)):
        overlap = all_pids_per_shard[i] & all_pids_per_shard[j]
        if overlap:
            print(f"⚠️  OVERLAP between {os.path.basename(shard_paths[i])} and "
                  f"{os.path.basename(shard_paths[j])}: {len(overlap)} shared problems")
            print(f"    example overlapping problem_ids: {sorted(overlap)[:10]}")
print("Overlap check complete.")

In [ ]:
# === Concatenate ===
X_parts, y_parts, pid_parts = [], [], []
for p, s in shards:
    X_parts.append(s["X"])
    y_parts.append(s["y"])
    pid_parts.append(s["problem_ids"])

X = torch.cat(X_parts, dim=0)
y = torch.cat(y_parts, dim=0)
pids = torch.cat(pid_parts, dim=0)

print(f"Merged X:           {tuple(X.shape)}  dtype={X.dtype}")
print(f"Merged y:           {tuple(y.shape)}  dtype={y.dtype}")
print(f"Merged problem_ids: {tuple(pids.shape)}  dtype={pids.dtype}")
print(f"Unique problems:    {len(torch.unique(pids))}")
print(f"Label distribution: {int(y.sum())} correct / {int((1 - y).sum())} incorrect "
      f"({y.mean().item():.1%} positive)")

In [ ]:
# === Save merged dataset + metadata ===
import json

torch.save(X, X_PATH)
torch.save(y, Y_PATH)
torch.save(pids, PID_PATH)

with open(MERGE_META_PATH, "w") as f:
    json.dump({
        "shards": rows,
        "total_positions": int(X.shape[0]),
        "hidden_dim": int(X.shape[1]),
        "unique_problems": int(len(torch.unique(pids))),
        "n_correct_positions": int(y.sum()),
        "n_incorrect_positions": int((1 - y).sum()),
    }, f, indent=2)

print(f"✓ X saved           -> {X_PATH}")
print(f"✓ y saved           -> {Y_PATH}")
print(f"✓ problem_ids saved -> {PID_PATH}")
print(f"✓ merge_meta saved  -> {MERGE_META_PATH}")

## Next step

- Open `probe_train.ipynb` (same folder) and run it.
- It will load `X.pt`, `y.pt`, `problem_ids.pt` from `OUTPUT_DIR`
- Train the linear probe with proper train/val/test splits and metrics (ROC-AUC, ECE, Brier). Runs on CPU in a few minutes, no compute units needed.